In [1]:
import warnings

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(split_name: str, feature_name: str) -> pd.DataFrame:
    path = f"datasets/{split_name}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(split_name: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(split_name, feature_names[0])
    for name in feature_names[1:]:
        extra = load_split(split_name, name)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        verbosity=0,
        early_stopping_rounds=50,
    )
    model.fit(
        train_df[feat_cols].values,
        train_df["target"].values,
        eval_set=[(val_df[feat_cols].values, val_df["target"].values)],
        verbose=False,
    )
    preds = model.predict(val_df[feat_cols].values)
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split("split1", feature_name)
    val1_df = load_split("split2", feature_name)
    val2_df = load_split("split3", feature_name)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": "split1_" + feature_name , "groups": feature_name, "rmsle": score1})
    results.append({"experiment": "split2_" + feature_name , "groups": feature_name, "rmsle": score2})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("split1", subset)
    val1_df = merge_splits("split2", subset)
    val2_df = merge_splits("split3", subset)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": f"split1_cumulative_{i}", "groups": label, "rmsle": score1})
    results.append({"experiment": f"split2_cumulative_{i}", "groups": label, "rmsle": score2})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))


 split1 RMSLE = 2.57133  (best iteration: 45) | split2 RMSLE = 2.77204  (best iteration: 33)

Ranking by RMSLE (lower = better)
             experiment                                                                                 groups    rmsle
             split1_gmv                                                                                    gmv 2.467789
           split1_trend                                                                                  trend 2.537059
    split1_cumulative_4                                        base + all_history + rolling_windows + velocity 2.552702
        split1_velocity                                                                               velocity 2.557468
    split1_cumulative_8 base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates + ewm 2.571332
    split1_cumulative_2                                                                     base + all_history 2.577245
     split1_all_history         

In [3]:
results = []

for i in range(2, len(FEATURES_GROUP)):
    if FEATURES_GROUP[i] != "gmv":
        subset = [FEATURES_GROUP[i], "gmv"]
        label = " + ".join(subset)
        train_df = merge_splits("split1", subset)
        val_df = merge_splits("split2", subset)
        score, best_iter = train_and_eval(train_df, val_df)
        results.append({"experiment": f"cumulative_{i}", "groups": label, "rmsle": score})

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))


Ranking by RMSLE (lower = better)
  experiment                 groups    rmsle
cumulative_6 conversion_rates + gmv 2.698405
cumulative_3         velocity + gmv 2.765246
cumulative_2  rolling_windows + gmv 2.784377
cumulative_4            trend + gmv 2.784919
cumulative_7              ewm + gmv 2.849100
